## Process Orders Data
- Injest the data into the data lakehouse -bronze_orders
- Perform data quality checks and transform the data as requiered - silver_orders_clean
- Explode the items array from the order object - silver_orders

In [0]:
--1
create or refresh streaming table bronze_orders
comment "Raw order data ingested from source system"
tblproperties ("quality" = "bronze")
as
select *,
    _metadata.file_path as input_file_path,
    current_timestamp as ingested_timestamp
    from cloud_files(
        '/Volumes/circuitbox/landing/operational_data/orders/',
        'json',
        map("cloudfiles.inferColumnTypes", "true")
    )


In [0]:
--2

create or refresh streaming table silver_orders_cleaned(
    constraint valid_customer_id expect (customer_id is not null) on violation fail update,
    constraint valid_order_id expect(order_id is not null) on violation fail update,
    constraint valid_order_status expect(order_status in ('Pending','Shipped', 'Completed', 'Cancelled')),
    constraint valid_payment_method expect(payment_method in ('Credit Card', 'Bank Transfer','PayPal'))
)
comment 'Cleaned order data'
tblproperties ("quality" = "silver")
as
select order_id,
    customer_id,
    order_status,
    CAST(order_timestamp as timestamp) as order_timestamp,
    payment_method
from stream(LIVE.bronze_orders)

In [0]:
--3
create or refresh streaming table silver_orders
as
select order_id,
    customer_id,
    order_status,
    order_timestamp,
    payment_method,
    item.item_id,
    item.name as item_name,
    item.price as item_price,
    item.quantity as item_quantity,
    item.category as item_category
from (
    select order_id,
        customer_id,
        order_status,
        order_timestamp,
        payment_method,
        explode(items) as item
    from stream(LIVE.bronze_orders)
)